# Notebook 01 — Data Cleaning: Dataset 1 (Training Set)

## Dataset used
This notebook cleans **Dataset 1**, which serves as the **training set** for all models.

The file loaded is `cosmetics_with_prop65.csv` from the folder `Prvi-Dataset-Cosmetic Safety and Hazardous Chemicals/`.

This file is a pre-merged version of the original CSCP dataset (`DatasetChemicalsInCosmetics.csv`) with the California Proposition 65 list, resulting in a `FlagProp65` column that serves as the **target label** for classification.

## Dataset split rationale
All three raw datasets originate from the same source — the California Safe Cosmetics Program (CSCP). Investigation revealed that D2 (CalHHS) and D3 (California Open Data Portal) are near-identical snapshots of the same database, with D2/D3 extending to June 2020 while D1 cuts off at February 2020.

The split strategy is therefore **temporal**:
- **Train:** Dataset 1 — records up to February 2020 (`cosmetics_with_prop65.csv`)
- **Test:** New records from D2/D3 — records from February to June 2020 (prepared in `02_cleaning_test.ipynb`)

_The dataset represents a temporal snapshot of CSCP reporting from 2009 to mid-2020; replication on more recent data is recommended as future work._

In [2]:
import pandas as pd

# Load D1 - already merged with Prop65
df = pd.read_csv('../data/raw/Prvi-Dataset-Cosmetic Safety and Hazardous Chemicals/cosmetics_with_prop65 (1).csv', low_memory=False)

print(f"Shape: {df.shape}")
print(f"\nColumns:\n{list(df.columns)}")
print("\nTarget label distribution:")
print(df['FlagProp65'].value_counts())
print(f"\nClass imbalance ratio: {df['FlagProp65'].value_counts()[False] / df['FlagProp65'].value_counts()[True]:.1f}:1")

Shape: (112870, 29)

Columns:
['CDPHId', 'ProductName', 'CSFId', 'CSF', 'CompanyId', 'CompanyName', 'BrandName', 'PrimaryCategoryId', 'PrimaryCategory', 'SubCategoryId', 'SubCategory', 'CasId', 'CasNumber', 'ChemicalId', 'ChemicalName', 'InitialDateReported', 'MostRecentDateReported', 'DiscontinuedDate', 'ChemicalCreatedAt', 'ChemicalUpdatedAt', 'ChemicalDateRemoved', 'ChemicalCount', 'InitialYear', 'FlagHighRisk', 'FlagCleanMarketed', 'CAS_norm', 'FlagProp65', 'Prop65_ChemicalName', 'Prop65_ToxicityType']

Target label distribution:
FlagProp65
False    107438
True       5432
Name: count, dtype: int64

Class imbalance ratio: 19.8:1


In [3]:
# data quality overview
print("=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing': missing, 'pct': missing_pct})
print(missing_df[missing_df['missing'] > 0].sort_values('pct', ascending=False))

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== DUPLICATE ROWS ===")
print(f"Full duplicates: {df.duplicated().sum()}")
print(f"Duplicate CDPHId+ChemicalName combos: {df.duplicated(subset=['CDPHId','ChemicalName']).sum()}")

print("\n=== SAMPLE OF TRADE SECRET ROWS (missing CasNumber) ===")
print(df[df['CasNumber'].isnull()][['ProductName','CompanyName','ChemicalName']].head(5))

=== MISSING VALUES ===
                     missing    pct
Prop65_ChemicalName   112870  100.0
Prop65_ToxicityType   112462   99.6
ChemicalDateRemoved   109949   97.4
DiscontinuedDate       99964   88.6
CSF                    34108   30.2
CSFId                  33683   29.8
CasNumber               6289    5.6
CAS_norm                6289    5.6
BrandName                223    0.2

=== DATA TYPES ===
CDPHId                      int64
ProductName                object
CSFId                     float64
CSF                        object
CompanyId                   int64
CompanyName                object
BrandName                  object
PrimaryCategoryId           int64
PrimaryCategory            object
SubCategoryId               int64
SubCategory                object
CasId                       int64
CasNumber                  object
ChemicalId                  int64
ChemicalName               object
InitialDateReported        object
MostRecentDateReported     object
DiscontinuedDate   

In [4]:
# === CLEANING STEP 1: Drop useless columns ===
# Prop65_ChemicalName is 100% empty - drop it
# Prop65_ToxicityType is 99.6% empty - drop it
# CSF and CSFId are free-text color/scent/flavor - not useful for ML
# ChemicalDateRemoved is 97.4% empty - drop it

cols_to_drop = ['Prop65_ChemicalName', 'Prop65_ToxicityType', 
                'CSF', 'CSFId', 'ChemicalDateRemoved']
df = df.drop(columns=cols_to_drop)
print(f"After dropping useless columns: {df.shape}")

# === CLEANING STEP 2: Drop full duplicate rows ===
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} full duplicate rows")

# === CLEANING STEP 3: Convert dates to datetime ===
date_cols = ['InitialDateReported', 'MostRecentDateReported', 
             'DiscontinuedDate', 'ChemicalCreatedAt', 'ChemicalUpdatedAt']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
print("\nDate columns converted")

# === CLEANING STEP 4: Handle missing BrandName ===
# 223 missing - fill with CompanyName as fallback
df['BrandName'] = df['BrandName'].fillna(df['CompanyName'])
print(f"BrandName nulls after fill: {df['BrandName'].isnull().sum()}")

# === CLEANING STEP 5: Flag trade secret rows (missing CasNumber) ===
# Don't drop them - trade secret itself is meaningful information
df['IsTradeSecret'] = df['CasNumber'].isnull().astype(int)
print(f"\nTrade secret rows: {df['IsTradeSecret'].sum()}")

# === CLEANING STEP 6: Handle DiscontinuedDate ===
# 88.6% missing means product is still active - fill with 0/1 flag
df['IsDiscontinued'] = df['DiscontinuedDate'].notnull().astype(int)
print(f"Discontinued products: {df['IsDiscontinued'].sum()}")

print(f"\nFinal shape: {df.shape}")
print(f"\nRemaining nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

After dropping useless columns: (112870, 24)
Dropped 49658 full duplicate rows

Date columns converted
BrandName nulls after fill: 0

Trade secret rows: 4428
Discontinued products: 9392

Final shape: (63212, 26)

Remaining nulls:
CasNumber            4428
DiscontinuedDate    53820
CAS_norm             4428
dtype: int64


In [6]:
# Understand what the duplicates were
df_original = pd.read_csv('../data/raw/Prvi-Dataset-Cosmetic Safety and Hazardous Chemicals/cosmetics_with_prop65 (1).csv', low_memory=False)
df_original = df_original.drop(columns=['Prop65_ChemicalName', 'Prop65_ToxicityType', 
                                         'CSF', 'CSFId', 'ChemicalDateRemoved'])

duplicates = df_original[df_original.duplicated(keep=False)]
print(f"Total rows involved in duplicates: {len(duplicates)}")
print("\nSample duplicate group:")
print(duplicates.head(4)[['CDPHId', 'ProductName', 'ChemicalName', 'CasNumber', 
                            'PrimaryCategory', 'InitialDateReported']].to_string())

# Check if duplicates are concentrated in certain products or chemicals
print("\nTop 5 products with most duplicates:")
print(duplicates['ProductName'].value_counts().head())

print("\nTop 5 chemicals with most duplicates:")
print(duplicates['ChemicalName'].value_counts().head())

Total rows involved in duplicates: 56200

Sample duplicate group:
    CDPHId                                          ProductName      ChemicalName   CasNumber PrimaryCategory InitialDateReported
6        7  ABSOLUTE Precision Color Powder System - All Shades  Titanium dioxide  13463-67-7   Nail Products          2009-07-13
7        7  ABSOLUTE Precision Color Powder System - All Shades  Titanium dioxide  13463-67-7   Nail Products          2009-07-13
8        7  ABSOLUTE Precision Color Powder System - All Shades  Titanium dioxide  13463-67-7   Nail Products          2009-07-13
12      12      ABSOLUTE FX Chromatones Precision Color Powders  Titanium dioxide  13463-67-7   Nail Products          2009-07-14

Top 5 products with most duplicates:
ProductName
Eyecolor                                                                               766
SEPHORA - COLORFUL OMBRE A PAUPIERES DECLIPSABLE / COLORFUL REMOVABLE EYESHADOW PAN    315
NYX GIRLS NAIL POLISH                               

In [8]:
# === CLEANING STEP 7: Create useful derived features ===

# Days since first reported (age of the product record)
reference_date = pd.Timestamp('2020-02-10')  # D1 cutoff date
df['DaysSinceReported'] = (reference_date - df['InitialDateReported']).dt.days

# Was the product ever updated after initial report?
df['WasUpdated'] = (df['MostRecentDateReported'] > df['InitialDateReported']).astype(int)

# How many days between first and most recent report (product activity span)
df['ReportingSpanDays'] = (df['MostRecentDateReported'] - df['InitialDateReported']).dt.days

print("=== NEW FEATURES ===")
print(df[['DaysSinceReported', 'WasUpdated', 'ReportingSpanDays']].describe())

# === CLEANING STEP 8: Drop columns not useful for ML ===
cols_to_drop_ml = [
    'ProductName', 'CompanyName', 'BrandName',  # free text
    'CasNumber', 'CAS_norm',                     # replaced by IsTradeSecret flag
    'ChemicalName',                              # high cardinality text
    'InitialDateReported', 'MostRecentDateReported',  # replaced by derived features
    'DiscontinuedDate', 'ChemicalCreatedAt', 'ChemicalUpdatedAt',  # raw dates
    'CDPHId', 'CompanyId', 'CasId', 'ChemicalId',  # ID columns
    'PrimaryCategoryId', 'SubCategoryId'         # redundant with text category cols
]

df_ml = df.drop(columns=cols_to_drop_ml)
print(f"\nML-ready shape: {df_ml.shape}")
print(f"\nFinal columns:\n{list(df_ml.columns)}")
print(f"\nRemaining nulls:\n{df_ml.isnull().sum()[df_ml.isnull().sum() > 0]}")

=== NEW FEATURES ===
       DaysSinceReported    WasUpdated  ReportingSpanDays
count       63212.000000  63212.000000       63212.000000
mean         2137.555923      0.437338         403.777574
std          1323.476653      0.496062         741.924474
min             0.000000      0.000000           0.000000
25%           921.000000      0.000000           0.000000
50%          1999.000000      0.000000           0.000000
75%          3542.000000      1.000000         449.000000
max          3890.000000      1.000000        3753.000000

ML-ready shape: (63212, 12)

Final columns:
['PrimaryCategory', 'SubCategory', 'ChemicalCount', 'InitialYear', 'FlagHighRisk', 'FlagCleanMarketed', 'FlagProp65', 'IsTradeSecret', 'IsDiscontinued', 'DaysSinceReported', 'WasUpdated', 'ReportingSpanDays']

Remaining nulls:
Series([], dtype: int64)


In [9]:
# === CLEANING STEP 9: Encode categorical columns ===
# PrimaryCategory and SubCategory are the only categoricals left

print("PrimaryCategory unique values:", df_ml['PrimaryCategory'].nunique())
print("SubCategory unique values:", df_ml['SubCategory'].nunique())

# Use pandas get_dummies (one-hot encoding)
df_encoded = pd.get_dummies(df_ml, columns=['PrimaryCategory', 'SubCategory'], dtype=int)

print(f"\nShape after encoding: {df_encoded.shape}")
print("\nFlagProp65 distribution (final):")
print(df_encoded['FlagProp65'].value_counts())
print(f"Class imbalance: {df_encoded['FlagProp65'].value_counts()[False] / df_encoded['FlagProp65'].value_counts()[True]:.1f}:1")

# === SAVE ===
df_encoded.to_csv('../data/processed/D1_cleaned.csv', index=False)
df.to_csv('../data/processed/D1_cleaned_readable.csv', index=False)  # version with text columns for EDA

print("\nSaved D1_cleaned.csv (ML-ready, encoded)")
print("Saved D1_cleaned_readable.csv (human-readable, for EDA)")

PrimaryCategory unique values: 13
SubCategory unique values: 89

Shape after encoding: (63212, 112)

FlagProp65 distribution (final):
FlagProp65
False    59815
True      3397
Name: count, dtype: int64
Class imbalance: 17.6:1

Saved D1_cleaned.csv (ML-ready, encoded)
Saved D1_cleaned_readable.csv (human-readable, for EDA)


Dataset 1 is fully cleaned and saved. Summary:

Started with 112,870 rows → ended with 63,212 after removing reporting duplicates
Dropped 5 useless columns (100% or 97% empty)
Created 3 new features: DaysSinceReported, WasUpdated, ReportingSpanDays
Created 2 flag features: IsTradeSecret, IsDiscontinued
One-hot encoded 13 primary categories and 89 subcategories → 112 final columns
Class imbalance: 17.6:1 (False:True) — to be handled in the modelling phase